# 06b · Qwen3-8B zero-shot (TabLLM)
## Tesis UBA 2026 — LLMs vs. ML Clásico para Credit Scoring

Primer experimento con el LLM: **zero-shot**, sin ejemplos. La lógica vive en
`scripts/06b_llm_prompting.py` (`run_llm_inference`). Usa **Ollama local** con `qwen3:8b`.

### ¿Qué medimos y por qué?
El zero-shot mide **cuánto sabe el LLM de credit scoring sin ver datos de esta cartera**,
usando solo su conocimiento preentrenado. Es la línea de base del enfoque LLM.

### ¿Cómo? — serialización TabLLM
Siguiendo Hegselmann et al. (2023), cada crédito se serializa como texto natural
(`descripción: valor`) con las 20 *features* más predictivas + las 4 variables de texto
libre. Los valores negativos del buró se serializan como **"sin historial"** para
preservar la semántica. El modelo responde una probabilidad de mora entera 0–100.

### Hipótesis del fracaso: el sesgo de selección
En el universo general, **mora histórica alta → más riesgo**. Pero este dataset es un
**portfolio ya aprobado**: a un cliente con mora alta solo se le aprobó el crédito si
compensaba con perfiles fuertes. Resultado: en estos datos la correlación se **invierte**
(r(wd81, target) = −0.40). El LLM aplica la heurística global correcta… que acá está
invertida. Esperamos AUC ≈ 0.50 (azar) o peor.

In [1]:
import importlib.util
from pathlib import Path

import pandas as pd
from IPython.display import Image


def _find_base() -> Path:
    base = Path.cwd()
    while not (base / "scripts").exists() and base != base.parent:
        base = base.parent
    return base


BASE = _find_base()


def load_script(filename: str):
    path = BASE / "scripts" / filename
    spec = importlib.util.spec_from_file_location("s_06b", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


m06b = load_script("06b_llm_prompting.py")

## Ejemplo de serialización TabLLM
Así "ve" el LLM a un crédito. Notar los `sin historial` (valores TU ausentes) y el texto
semántico del negocio al final.

In [2]:
df = m06b.load_segmented(str(BASE / "data" / "dataset_tesis.csv"))
ejemplo = df.iloc[0]
serializado = m06b.serialize_tabllm(ejemplo, m06b.TOP_FEATURES_LLM)
print(serializado[:700], "...")

el número de sectores económicos distintos con reporte activo en el buró: 20.00, la peor calificación de mora histórica — máximo días en atraso (últimos 81 meses): 8.00, la peor calificación de mora reciente — máximo días en atraso (últimos 12 meses): 8.00, la antigüedad promedio de las cuentas activas en el buró en meses: sin historial, el número de consultas al buró en los últimos 6 meses: sin historial, el número de cuentas en mora actualmente: sin historial, el porcentaje de créditos pagados a tiempo en los últimos 4 años: sin historial, el número de cuentas activas reportadas en el buró: 20.00, el saldo total de deudas vigentes en el buró (últimos 8 meses) en pesos: 176.00, la utilizaci ...


## Prompt completo enviado a Ollama

In [3]:
msgs = m06b._build_messages(ejemplo, m06b.TOP_FEATURES_LLM, think=False)
print("[SYSTEM]\n" + msgs[0]["content"])
print("\n[USER] (final)\n... " + msgs[1]["content"][-200:])

[SYSTEM]
Eres evaluador de riesgo crediticio de microcréditos en Colombia. Responde SOLO con un número entero.

[USER] (final)
... jetivo crédito: Pago a proveedores, educación: No especificado, canal: CANAL 09, subcategoría: arreglos, negocio: Moda y accesorios, tipo crédito: Primer Crédito

Probabilidad de mora (0-100, entero):


## Ejecución (reutiliza el cache de inferencia ya calculado)
La inferencia sobre las 5.351 filas ya está cacheada en `models/llm_probs_tabllm_nothink.parquet`,
por lo que esta celda solo **evalúa** sobre el conjunto de test. Evaluamos por segmento.

In [4]:
res = m06b.run_llm_inference(
    dataset_path=str(BASE / "data" / "dataset_tesis.csv"),
    cache_path=str(BASE / "models" / "llm_probs_tabllm_nothink.parquet"),
    mode="nothink",
    output_dir=str(BASE / "models"),
)
filas = [{"segmento": seg, **{k: round(v, 3) for k, v in res["test"][seg].items()
                              if isinstance(v, (int, float))}}
         for seg in ("total", "esparso", "denso")]
pd.DataFrame(filas)

,segmento,AUC,Gini,KS,Brier,PR_AUC,n
0,total,0.529,0.058,0.041,0.375,0.507,495
1,esparso,0.460,-0.080,0.059,0.448,0.387,28
2,denso,0.533,0.067,0.046,0.370,0.515,467


**Lectura.** El AUC total ≈ 0.53 — casi aleatorio. Confirma **experimentalmente** el
sesgo de selección: el conocimiento global del LLM, correcto en general, es
contraproducente en un portfolio aprobado. No es un *bug*; es un *failure mode*
caracterizable. El few-shot (notebook 08) intenta corregirlo mostrándole ejemplos reales.